In [13]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

RAW_DIR = Path("../data/raw")
EXPECTED_FILES = {
    "transactions": RAW_DIR / "transactions_data.csv",
    "labels": RAW_DIR / "train_fraud_labels.json",
    "users": RAW_DIR / "users_data.csv",
    "cards": RAW_DIR / "cards_data.csv",
    "mcc": RAW_DIR / "mcc_codes.json",
}

file_check = pd.DataFrame(
    {"file": [path.name for path in EXPECTED_FILES.values()], "exists": [path.exists() for path in EXPECTED_FILES.values()]}
)
file_check

,file,exists
0,transactions_data.csv,True
1,train_fraud_labels.json,True
2,users_data.csv,True
3,cards_data.csv,True
4,mcc_codes.json,True


In [14]:
required = [EXPECTED_FILES["transactions"], EXPECTED_FILES["labels"]]
missing_required = [path.name for path in required if not path.exists()]
if missing_required:
    raise FileNotFoundError(f"Download these files into {RAW_DIR}: {missing_required}")

transactions = pd.read_csv(EXPECTED_FILES["transactions"])
users = pd.read_csv(EXPECTED_FILES["users"]) if EXPECTED_FILES["users"].exists() else None
cards = pd.read_csv(EXPECTED_FILES["cards"]) if EXPECTED_FILES["cards"].exists() else None

with EXPECTED_FILES["labels"].open() as handle:
    raw_labels = json.load(handle)
label_mapping = raw_labels.get("target", raw_labels)
labels = pd.DataFrame(label_mapping.items(), columns=["id", "is_fraud"])
labels["id"] = pd.to_numeric(labels["id"], errors="coerce")
labels = labels.dropna(subset=["id"]).astype({"id": "int64"})
labels["is_fraud"] = labels["is_fraud"].replace({"Yes": 1, "No": 0, "1": 1, "0": 0})
labels["is_fraud"] = pd.to_numeric(labels["is_fraud"], errors="coerce")

dataset_sizes = pd.DataFrame(
    [("transactions", *transactions.shape), ("labels", *labels.shape)]
    + ([] if users is None else [("users", *users.shape)])
    + ([] if cards is None else [("cards", *cards.shape)]),
    columns=["dataset", "rows", "columns"],
)
dataset_sizes

/var/folders/8z/cq714m8948vbg8qf9wfmx_p40000gn/T/ipykernel_41008/3124824216.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  labels["is_fraud"] = labels["is_fraud"].replace({"Yes": 1, "No": 0, "1": 1, "0": 0})


,dataset,rows,columns
0,transactions,13305915,12
1,labels,8914963,2
2,users,2000,14
3,cards,6146,13


In [15]:
def profile_frame(name, frame):
    return pd.DataFrame(
        {
            "dataset": name,
            "column": frame.columns,
            "dtype": frame.dtypes.astype(str).values,
            "missing_count": frame.isna().sum().values,
            "missing_percent": (frame.isna().mean() * 100).round(2).values,
            "unique_count": frame.nunique(dropna=True).values,
        }
    )

profiles = [profile_frame("transactions", transactions), profile_frame("labels", labels)]
if users is not None:
    profiles.append(profile_frame("users", users))
if cards is not None:
    profiles.append(profile_frame("cards", cards))

column_profile = pd.concat(profiles, ignore_index=True)
column_profile.sort_values(["dataset", "missing_percent"], ascending=[True, False])

,dataset,column,dtype,missing_count,missing_percent,unique_count
28,cards,id,int64,0,0.00,6146
29,cards,client_id,int64,0,0.00,2000
30,cards,card_brand,object,0,0.00,4
31,cards,card_type,object,0,0.00,3
32,cards,card_number,int64,0,0.00,6146
33,cards,expires,object,0,0.00,259
34,cards,cvv,int64,0,0.00,998
35,cards,has_chip,object,0,0.00,2
36,cards,num_cards_issued,int64,0,0.00,3
37,cards,credit_limit,object,0,0.00,3654


In [16]:
quality_checks = pd.DataFrame(
    {
        "check": [
            "transaction duplicate rows",
            "transaction duplicate ids",
            "label duplicate ids",
            "labels with missing fraud target",
        ],
        "value": [
            int(transactions.duplicated().sum()),
            int(transactions["id"].duplicated().sum()) if "id" in transactions else np.nan,
            int(labels["id"].duplicated().sum()),
            int(labels["is_fraud"].isna().sum()),
        ],
    }
)
quality_checks

,check,value
0,transaction duplicate rows,0
1,transaction duplicate ids,0
2,label duplicate ids,0
3,labels with missing fraud target,0


In [17]:
amount_text = transactions["amount"].astype(str) if "amount" in transactions else pd.Series(dtype="object")
amount_numeric = pd.to_numeric(amount_text.str.replace("$", "", regex=False), errors="coerce")
date_column = "date" if "date" in transactions else "timestamp" if "timestamp" in transactions else None
parsed_dates = pd.to_datetime(transactions[date_column], errors="coerce") if date_column else pd.Series(dtype="datetime64[ns]")

parse_checks = pd.DataFrame(
    {
        "check": ["invalid numeric amount", "invalid transaction date"],
        "value": [int(amount_numeric.isna().sum()), int(parsed_dates.isna().sum())],
    }
)
parse_checks

,check,value
0,invalid numeric amount,0
1,invalid transaction date,0


In [18]:
merged = transactions.merge(labels.dropna(subset=["is_fraud"]), on="id", how="inner")
label_summary = merged["is_fraud"].value_counts(dropna=False).rename_axis("is_fraud").reset_index(name="rows")
label_summary["percent"] = (label_summary["rows"] / len(merged) * 100).round(2)
label_summary

,is_fraud,rows,percent
0,0,8901631,99.85
1,1,13332,0.15
